# TASK 1

Ingestion, preprocess and embeddings of the course materials.
We use PySpark to read PDF documents, extract and clean the text, split it into chunks and create embeddings with either TF-IDF, SBERT or Word2Vec. The resulting chunks and their embeddings are saved in parquet format for the next notebook.

In [1]:
"""
import sys
!pip install --upgrade transformers sentence-transformers

!{sys.executable} -m pip install --upgrade --force-reinstall --no-cache-dir transformers sentence-transformers

 On met à jour Pandas pour qu'il s'aligne avec le reste
!{sys.executable} -m pip install --upgrade pandas "numpy<2.0"

!{sys.executable} -m pip install gensim
"""

import re
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict



# we use this library to extract the text out of the pdf
from pypdf import PdfReader

from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession 

# Configuration of the rag pipeline
Instead of scattering our parameters everywhere in the notebook, we centralize everything in one dataclass. This is easier and faster when you want to test another embedding strategy or change chunk size, because you only need to change your settings in this cell 

In [2]:
@dataclass
class RAGConfig:
    # paths for storing the pdf results of embeddings
    input_dir: str = "course_material"
    output_dir: str = "/home/guest/shared/course_material"
    parallelism: int = 4 

    #  Chunking 
    chunk_size_words: int = 200
    chunk_overlap_words: int = 50

    #  Embedding
    embedding_strategy: str = "sbert"   # {"tfidf", "sbert", "word2vec"} 
    sbert_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"
cfg = RAGConfig()

# Spark Initialization
We are using spark to treat several pdf's in parallel thanks of its processes which optimizes the use of cpu cores.

In [3]:
# Init Spark
if SparkContext._active_spark_context:
    SparkContext._active_spark_context.stop()
    print("Existing SparkContext stopped.")
    
conf = SparkConf() \
    .setAppName("IngestData") \
    .setMaster("local[*]")

sc = SparkContext(conf=conf)
print("Spark version:", sc.version)

Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/21 19:20:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.2.1


# PDF Ingestion
We retrieve the list with every pdf files in the configured folder, then we give it to spark in rdd. Every element of the RDD is a path to a pdf. Spark will then treat then in parallel.

In [4]:
def get_paths_pdf_files(folder_path: str) -> List[str]:
    """
    Browse a folder and get path from each PDF files.

    Parameters
    ----------
    folder_path : str
        Path to the folder containing the PDF files.

    Returns
    -------
    results : list of string
    """

    folder = Path(folder_path)

    # return paths all pdf files in folder path
    return list( folder.glob("*.pdf") )

#  quick sanity check (should not error once implemented) 
pdf_paths = get_paths_pdf_files(cfg.input_dir)
rdd = sc.parallelize(pdf_paths, numSlices=cfg.parallelism)
print(pdf_paths)

[PosixPath('course_material/1-intro.pdf'), PosixPath('course_material/2-spark.pdf'), PosixPath('course_material/3-stream_proc.pdf'), PosixPath('course_material/4-stream_algos.pdf'), PosixPath('course_material/5-nosql.pdf'), PosixPath('course_material/6-parallel.pdf'), PosixPath('course_material/9780521192248_excerpt.pdf'), PosixPath('course_material/A Neural Network Playground.pdf'), PosixPath('course_material/a_Big Data, Data Mining, and Machine Learning ( PDFDrive.com )-min.pdf'), PosixPath('course_material/a_ScalableMachineLearningAlgorithmsforBigDataAnalytics-ChallengesandOpportunities.pdf'), PosixPath('course_material/a_the-big-book-of-machine-learning-use-cases.pdf'), PosixPath('course_material/a_TowardScalableSystemsForBigDataAnalytics.pdf'), PosixPath('course_material/Finding Similar Items.pdf'), PosixPath('course_material/Gama_ACMCS_AdaptationCD_accepted (1).pdf'), PosixPath('course_material/Gama_ACMCS_AdaptationCD_accepted.pdf'), PosixPath('course_material/intro.pdf'), PosixP

# Page by page extraction of the text
Definition of the function which will open a pdf and extract its text. This function will be applied in parallel on every pdf thanks to spark.

The flatmap at the end flattened the list of pages of each pdf in one RDD where each element is one individual page

In [5]:
def extract_pages_from_pdf(pdf_path: str) -> List[Dict]:
    """
    Exctraction of the text of a pdf page by page.
    
    :param pdf_path: path to the pdf file 
    :return: a list of dict of text extracted from the pdf 
    """
    
    import logging
    logging.getLogger("pypdf").setLevel(logging.ERROR)
    # Read pdf file
    reader = PdfReader(pdf_path)

    # For each page
    pages = list()
    for i in range( len(reader.pages) ):
        # Extract the text
        page = {
            "pdf_name" : Path(pdf_path).name,
            "page_num" : i + 1,
            "text" : reader.pages[i].extract_text() or ""
        }
        # Add the text to the list of pages
        pages.append(page)

    return pages

pages_rdd = rdd.flatMap(extract_pages_from_pdf)

# Cleaning of the text
Pypdf reconstruct the text line by line from the pdf. This creates artefacts (words that are cut at the end of a line, line jumps in the middle of a sentence, multiple spaces between some words).
We introduce 4 cleaning steps:
1) pypdf removes words that containes end of a line hyphens
2) it replaces the remaining line breaks with space
3) then it reduce multiple spaces to a single space
4) finally it remove leading and trailing spaces

In [6]:
def clean_text(page_from_pdf: dict) -> dict:
    """Basic cleanup for extracted pdf text.
    
    input : dict with { "pdf_name", "page_num", "text" }
    output : dict with the same keys but the text is cleaned
    
    """
    text = page_from_pdf["text"]

    text = text.replace("-\n", "")
    
    
    text = text.replace("\n", " ")
    
    
    text = re.sub(r"\s+", " ", text)
    
    
    page_from_pdf["text"] = text.strip()
    
    return page_from_pdf

# map = application of the cleaning process on every page
clean_rdd = pages_rdd.map(clean_text)
print(clean_rdd.count(), "extracted pages")
print(clean_rdd.first())

1662 extracted pages


{'pdf_name': '1-intro.pdf', 'page_num': 1, 'text': 'INFO-H515: BIG DATA: DISTRIBUTED MANAGEMENT Lecture 1: Introduction, HDFS, Map/Reduce Dimitris Sacharidis'}


# Splitting into chunks
Each chunk is a rolling window of a number (chunk_size_words) of words. The overlap_words guarantees that if a concept is on more than one chunk, it won't be lost. Each chunk keeps note of its position in the page to be able to find back its source.

In [7]:
def word_based_chunk_page(page_from_pdf: dict) -> list:
    """Split one page into overlapping word chunks.
    
    
    input : dict with { "pdf_name", "page_num", "text" }
    output : list of dict with { "chunk_id", "pdf_name", "page_num", "text", "start_word", "end_word" }
    
    """
    
    chunk_size_words = cfg.chunk_size_words 
    overlap_words = cfg.chunk_overlap_words
    words = page_from_pdf["text"].split()
    page_num = page_from_pdf["page_num"]    
    chunks = []
    step = chunk_size_words - overlap_words
    if step <= 0:
        raise ValueError("overlap_words must be smaller than chunk_size_words")
    chunk_index = 0
    for start in range(0, len(words), step):
        end = start + chunk_size_words
        chunk_words = words[start:end]
        if not chunk_words:
            break
        chunks.append({
            "chunk_id": f"p{page_num}_c{chunk_index}",
            "pdf_name": page_from_pdf["pdf_name"],
            "page_num":  page_num,
            "text": " ".join(chunk_words),
            "start_word": start,        
            "end_word": end - 1
        })
        chunk_index += 1
        if end >= len(words):
            break
    return chunks



# flatmap allows every page to become a list of chunks, we flatten it in one RDD
clean_chunk_rdd = clean_rdd.flatMap(word_based_chunk_page)
print(clean_chunk_rdd.count(), " chunks")
print(clean_chunk_rdd.first())

2982  chunks


{'chunk_id': 'p1_c0', 'pdf_name': '1-intro.pdf', 'page_num': 1, 'text': 'INFO-H515: BIG DATA: DISTRIBUTED MANAGEMENT Lecture 1: Introduction, HDFS, Map/Reduce Dimitris Sacharidis', 'start_word': 0, 'end_word': 199}


In [8]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Creation of the embeddings pipeline
We setup a pipeline for our embeddings. We transform every text chunk in embedding vectors so that they can be processed by the retriever in Notebook 2.

We use TF-IDF, SBERT and WOrd2Vec. We create a function sfg.embedding_strategy for our common pipeline.

In [9]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, FloatType, ArrayType
from pyspark.ml.feature import Tokenizer, HashingTF, IDF, Word2Vec as SparkWord2Vec
from pyspark.sql.functions import udf, lit

# We reuse the sparkcontext that we have already created
spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()




#    then we create the dataframe with the id's and metadata
rdd_base = clean_chunk_rdd.zipWithIndex().map(
    lambda x: (
        x[1],                        # id (LongType)
        x[0]["pdf_name"],            # source_pdf
        x[0]["page_num"],            # page_num
        x[0]["chunk_id"],            # chunk_id
        x[0]["start_word"],          # start_word (Entier)
        x[0]["end_word"],            # end_word (Entier)
        x[0]["text"],                # chunk_text
    )
)

schema_base = StructType([
    StructField("id", LongType(), False),
    StructField("source_pdf", StringType(), True),
    StructField("page_num", IntegerType(), True),
    StructField("chunk_id", StringType(), True),
    StructField("start_word", IntegerType(), True),
    StructField("end_word", IntegerType(), True),
    StructField("chunk_text", StringType(), True),
])

df_base = spark.createDataFrame(rdd_base, schema=schema_base)

# final schema with embeddings columns
final_schema = StructType(schema_base.fields + [
    StructField("embedding", ArrayType(FloatType()), True),
    StructField("embedding_strategy", StringType(), True)
])

df_final = None # 


# different embedding strategies


if cfg.embedding_strategy == "sbert":
    # 1ST STRATEGY : SBERT 
    def embed_partition_sbert(partition):
        from sentence_transformers import SentenceTransformer
        import numpy as np
        
        model = SentenceTransformer(cfg.sbert_model_name)
        for row in partition:
            row_dict = row.asDict()
            emb = model.encode(row_dict["chunk_text"], normalize_embeddings=True)
            row_dict["embedding"] = np.asarray(emb, dtype=np.float32).tolist()
            row_dict["embedding_strategy"] = "sbert"
            yield row_dict

    rdd_embedded = df_base.rdd.mapPartitions(embed_partition_sbert)
    df_final = spark.createDataFrame(rdd_embedded, schema=final_schema)

elif cfg.embedding_strategy == "tfidf":
    # 2ND STRATEGY : TF-IDF 
    tokenizer = Tokenizer(inputCol="chunk_text", outputCol="words")
    wordsData = tokenizer.transform(df_base)
    
    hashingTF = HashingTF(inputCol="words", outputCol="rawFeatures", numFeatures=20000)
    featurizedData = hashingTF.transform(wordsData)
    
    idf = IDF(inputCol="rawFeatures", outputCol="features")
    idfModel = idf.fit(featurizedData)
    rescaledData = idfModel.transform(featurizedData)
    ## Spark stores vectors in sparse format, then we convert them to Python lists
    # so we can save them in parquet format and easily read them back into Notebook 2
    vector_to_array = udf(lambda v: v.toArray().tolist(), ArrayType(FloatType()))
    
    df_final = rescaledData.withColumn("embedding", vector_to_array("features")) \
                           .withColumn("embedding_strategy", lit("tfidf")) \
                           .drop("words", "rawFeatures", "features")
    idfModel.write().overwrite().save(f"{cfg.output_dir}/tfidf_model.model")
elif cfg.embedding_strategy == "word2vec":
    # 3RD STRATEGY : Word2Vec 
    tokenizer = Tokenizer(inputCol="chunk_text", outputCol="words")
    wordsData = tokenizer.transform(df_base)
    
    word2Vec = SparkWord2Vec(vectorSize=100, minCount=2, inputCol="words", outputCol="features")
    model = word2Vec.fit(wordsData)
    w2vData = model.transform(wordsData)
    
    vector_to_array = udf(lambda v: v.toArray().tolist(), ArrayType(FloatType()))
    
    df_final = w2vData.withColumn("embedding", vector_to_array("features")) \
                      .withColumn("embedding_strategy", lit("word2vec")) \
                      .drop("words", "features")

    model.write().overwrite().save(f"{cfg.output_dir}/w2v_model.model")
    

# BACKUP AND TEST

if df_final is not None:
    import os
    linux_internal_path = f"/home/guest/embeddings_{cfg.embedding_strategy}.parquet"
    df_final.write.mode("overwrite").parquet(linux_internal_path)
    print(f" Succesfully saved : {linux_internal_path}")
    
    shared_path = f"{cfg.output_dir}/embeddings_{cfg.embedding_strategy}.parquet"
    os.system(f"cp -r {linux_internal_path} {shared_path}")
    print(f" files copied on the PC : {shared_path}\n")
    
    # Reading test
    print("Verification of the parquet file :")
    test_df = spark.read.parquet(shared_path)
    print(f"Number of chunks saved : {test_df.count()}")
    test_df.select("id", "source_pdf", "chunk_id", "start_word", "end_word", "embedding_strategy").show(3)

/home/guest/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/guest/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (
/home/guest/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/guest/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALL

 Succesfully saved : /home/guest/embeddings_sbert.parquet
 files copied on the PC : /home/guest/shared/course_material/embeddings_sbert.parquet

Verification of the parquet file :


Number of chunks saved : 2982
+---+--------------------+--------+----------+--------+------------------+
| id|          source_pdf|chunk_id|start_word|end_word|embedding_strategy|
+---+--------------------+--------+----------+--------+------------------+
|627|A Neural Network ...|   p1_c0|         0|     199|             sbert|
|628|A Neural Network ...|   p1_c1|       150|     349|             sbert|
|629|A Neural Network ...|   p2_c0|         0|     199|             sbert|
+---+--------------------+--------+----------+--------+------------------+
only showing top 3 rows



# Inspection of the parquet file
We list and manually inspect

In [10]:
import os
chemin = f"{cfg.output_dir}/embeddings_{cfg.embedding_strategy}.parquet"
print("Files in the output directory:")
print(os.listdir(chemin))

Files in the output directory:
['.part-00000-912e4fb2-94df-4af6-b303-25a85d14899b-c000.snappy.parquet.crc', '.part-00001-912e4fb2-94df-4af6-b303-25a85d14899b-c000.snappy.parquet.crc', '.part-00002-912e4fb2-94df-4af6-b303-25a85d14899b-c000.snappy.parquet.crc', '.part-00003-912e4fb2-94df-4af6-b303-25a85d14899b-c000.snappy.parquet.crc', '._SUCCESS.crc', 'embeddings_sbert.parquet', 'part-00000-912e4fb2-94df-4af6-b303-25a85d14899b-c000.snappy.parquet', 'part-00001-912e4fb2-94df-4af6-b303-25a85d14899b-c000.snappy.parquet', 'part-00002-912e4fb2-94df-4af6-b303-25a85d14899b-c000.snappy.parquet', 'part-00003-912e4fb2-94df-4af6-b303-25a85d14899b-c000.snappy.parquet', '_SUCCESS']
